#### **Regression Modeling & Hypothesis Testing**

**Goal:**  
Build and interpret statistical models to quantify relationships and formally test key hypotheses.

**Key Hypotheses to Test:**
1. Sustainability features are associated with a statistically significant positive premium on price per sqft (expected 8–12%).
2. First-Time buyers transact at significantly lower average prices than Investors (after controlling for other factors).
3. Dubai properties have higher price per sqft than Abu Dhabi (controlling for property type, size, etc.).
4. Price per sqft increased significantly over 2023–2025, with stronger growth in Dubai.
5. Off-plan properties show different pricing patterns (potentially premium or discount depending on market phase).

**Approach:**
- Primary model: OLS linear regression on `price_per_sqft` (log-transformed for better fit in luxury markets)
- Control variables: size, property_type, emirate, year (or time dummies), off-plan, etc.
- Use statsmodels for detailed summary tables (coefficients, p-values, R²)
- Dummy encoding for categorical variables

**Input:** `../data/cleaned_transactions_2023_2025.csv`  
**Output:** Model summaries, coefficient interpretations, hypothesis conclusions

#### **1. Imports & Setup**

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print("Modeling libraries ready.")

Modeling libraries ready.


#### **2. Load Data & Prepare for Modeling**

In [2]:
INPUT_PATH = '../data/cleaned_transactions_2023_2025.csv'
df = pd.read_csv(INPUT_PATH, parse_dates=['date'])

print(f"Loaded {len(df):,} rows")

# Create log price per sqft (common in real estate to handle skewness)
df['log_price_per_sqft'] = np.log(df['price_per_sqft'])

# Ensure categoricals are properly typed
df['emirate'] = df['emirate'].astype('category')
df['property_type'] = df['property_type'].astype('category')
df['buyer_type'] = df['buyer_type'].astype('category')
df['year'] = df['year'].astype('category')  # treat as categorical for dummies

# Drop any rows with invalid log (price <=0 — shouldn't happen)
df = df[df['price_per_sqft'] > 0].copy()

print("\nReady for modeling. Log-transformed target created.")

Loaded 4,000 rows

Ready for modeling. Log-transformed target created.


#### **3. Model 1 – Basic Sustainability & Buyer Type Impact**

In [3]:
formula1 = (
    'log_price_per_sqft ~ C(sustainability_flag) + C(buyer_type) + C(emirate) + '
    'C(year) + size_sqft + C(property_type) + C(is_offplan)'
)

model1 = smf.ols(formula1, data=df).fit()

print(model1.summary().tables[1]) 
print(f"\nR-squared: {model1.rsquared:.4f}")
print(f"Adjusted R-squared: {model1.rsquared_adj:.4f}")
print(f"Observations: {model1.nobs:,}")

                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                          7.0341      0.018    396.773      0.000       6.999       7.069
C(sustainability_flag)[T.True]     0.0914      0.012      7.640      0.000       0.068       0.115
C(buyer_type)[T.Investor]          0.0550      0.012      4.481      0.000       0.031       0.079
C(buyer_type)[T.Other]             0.0170      0.017      1.002      0.316      -0.016       0.050
C(emirate)[T.Dubai]                0.2797      0.012     22.842      0.000       0.256       0.304
C(year)[T.2024]                    0.1253      0.013      9.397      0.000       0.099       0.151
C(year)[T.2025]                    0.1863      0.013     14.043      0.000       0.160       0.212
C(property_type)[T.Townhouse]      0.0503      0.020      2.482      0.013       0.011       0.090
C(property

#### **Model 1 Key Findings & Interpretation**

**Model Summary:**
- Dependent variable: log(price_per_sqft) → coefficients ≈ percentage change in price per sqft
- R-squared = 0.1792 (Adjusted 0.1771) → modest explanatory power (typical in real estate models without location-area fixed effects or more granular controls; still useful for directional insights)
- Observations: 4,000

**Statistically Significant Coefficients (p < 0.05):**

- **Sustainability Flag (True)**: coef = 0.0914 (p = 0.000)  
  → Properties with sustainability features command 9.1% higher price per sqft (exp(0.0914) - 1 ≈ 9.56%), holding other factors constant.  
  Strongly supports H1: clear, significant premium for green/livable features.

- **Buyer Type – Investor** (vs First-Time baseline): coef = 0.0550 (p = 0.000)  
  → Investors pay 5.7% more per sqft than First-Time buyers (after controlling for size, type, emirate, year, etc.).  
  → Other category not significant (p = 0.316).  
  Partially supports H2: investors do pay a premium, though smaller than raw averages suggested (43% higher mean price) — much of the gap is explained by property characteristics.

- **Emirate – Dubai** (vs Abu Dhabi baseline): coef = 0.2797 (p = 0.000)  
  → Dubai properties command 32.3% higher price per sqft (exp(0.2797) - 1 ≈ 32.3%), even after controls.  
  Strongly confirms H3: large structural premium in Dubai.

- **Year dummies** (vs 2023 baseline):  
  - 2024: coef = 0.1253 (p = 0.000) → 13.3% higher  
  - 2025: coef = 0.1863 (p = 0.000) → 20.5% higher  
  → Clear upward time trend, with cumulative 20.5% increase by 2025 — supports H4.

- **Property Type – Townhouse** (vs Apartment baseline): coef = 0.0503 (p = 0.013)  
  → 5.2% premium for townhouses.

- **Non-significant** (p > 0.05):  
  - Villas (no difference vs apartments)  
  - Off-plan (no consistent premium/discount after controls)  
  - Size_sqft (surprisingly flat effect — likely because price_per_sqft already normalizes size; model could be re-run on total price_aed if needed)

**Business & Portfolio Takeaways:**
- **Sustainability premium quantified at 9%** — actionable insight: developers can justify higher pricing or better margins by incorporating green features (especially in Dubai where base prices are higher).
- **First-Time buyers remain the price-sensitive segment** — even after controls, they transact at lower effective values → ideal target for mid-range, government-supported programs.
- **Time trend strong** → market still appreciating, but slowing (2025 growth lower than 2024) → opportunity to position projects now.
- **Model limitation**: Low R² suggests missing granular location (area-level) or demand-side variables — future iterations could add C(area) or interaction terms.

These results provide data-backed evidence to support the core project recommendation:  
Target mid-income first-time buyers with sustainable, livable mid-tier developments in Dubai → leverage ~9% pricing uplift from sustainability + strong time/emirate trends for 10–15% potential ROI improvement.

#### 4. Model 2 – Interaction: Sustainability × Emirate

In [7]:
formula2 = (
    'log_price_per_sqft ~ C(sustainability_flag) * C(emirate) + C(buyer_type) + '
    'C(year) + size_sqft + C(property_type) + C(is_offplan)'
)

model2 = smf.ols(formula2, data=df).fit()

print(model2.summary().tables[1])
print(f"\nR-squared: {model2.rsquared:.4f}")

                                                         coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------------
Intercept                                              7.0293      0.018    380.500      0.000       6.993       7.066
C(sustainability_flag)[T.True]                         0.1101      0.024      4.675      0.000       0.064       0.156
C(emirate)[T.Dubai]                                    0.2863      0.014     20.140      0.000       0.258       0.314
C(buyer_type)[T.Investor]                              0.0553      0.012      4.502      0.000       0.031       0.079
C(buyer_type)[T.Other]                                 0.0173      0.017      1.018      0.309      -0.016       0.051
C(year)[T.2024]                                        0.1252      0.013      9.385      0.000       0.099       0.151
C(year)[T.2025]                                 

#### **Model 2 Key Findings & Interpretation**

**Model Summary:**
- Dependent variable: log(price_per_sqft) → coefficients interpretable as approximate % changes
- R-squared = 0.1794 (very similar to Model 1) → adding the interaction term did not meaningfully improve explanatory power
- Observations: 4,000
- Interaction term included: sustainability_flag × emirate

**Statistically Significant Coefficients (p < 0.05):**

- **Sustainability Flag (True)** (main effect, for Abu Dhabi baseline): coef = 0.1101 (p = 0.000)  
  → In Abu Dhabi, properties with sustainability features command 11.6% higher price per sqft (exp(0.1101) - 1 ≈ 11.64%).

- **Emirate – Dubai** (baseline Abu Dhabi): coef = 0.2863 (p = 0.000)  
  → Dubai properties are 33.1% more expensive per sqft (exp(0.2863) - 1 ≈ 33.1%), after controls.

- **Buyer Type – Investor** (vs First-Time): coef = 0.0553(p = 0.000)  
  → Investors pay 5.7% more per sqft than First-Time buyers.

- **Year dummies** (vs 2023):  
  - 2024: coef = 0.1252 (p = 0.000) → 13.3% higher  
  - 2025: coef = 0.1862 (p = 0.000) → 20.4% higher  
  → Consistent upward trend.

- **Property Type – Townhouse** (vs Apartment): coef = 0.0496 (p = 0.014)  
  → 5.1% premium.

**Key Non-significant Results:**
- **Interaction: Sustainability × Dubai** → coef = -0.0252 (p = 0.358)  
  → The sustainability premium does not differ significantly between Dubai and Abu Dhabi (interaction p > 0.05).  
  → The premium is similar across both emirates (9–11% range when combining main + interaction effects).

- Off-plan, size_sqft, and Other buyer type remain non-significant (consistent with Model 1).

**Combined Interpretation of Sustainability Premium (from Model 2):**
- Abu Dhabi: 11.6% premium  
- Dubai: 11.6% + (-2.5%) interaction ≈ 9.1% premium  
→ Very close to the raw averages from Phase 4 (Abu Dhabi 12.3%, Dubai 8.7%) and Model 1's overall 9.1%.  
→ The difference between emirates is not statistically significant — sustainability adds consistent value (9–12%) regardless of emirate.

**Business & Portfolio Takeaways:**
- The sustainability premium is robust and statistically significant (9–12% on price per sqft) but does not vary meaningfully by emirate → a strong, general recommendation: incorporate green/livable/sustainable features in new projects to support higher pricing or better margins.
- No evidence that sustainability is more (or less) rewarded in Dubai vs Abu Dhabi → strategy can be applied uniformly across both markets.
- Reinforces targeting first-time buyers (lower effective prices) with sustainable mid-range developments, especially in Dubai where the overall price level is 33% higher.
- Model limitation: Low R² (0.18) indicates that area-level granularity (e.g., Palm Jumeirah vs JVC) or additional demand factors would improve fit — consider in future iterations.

This model strengthens the core recommendation:  
Emphasize sustainability in project positioning → expect ~9–12% pricing uplift, supporting 10–15% potential ROI improvement when targeting mid-income first-time buyers via government-backed programs.

#### **5. Model 3 – Focus on First-Time Buyers & Mid-Range**

In [6]:
formula3 = (
    'log_price_per_sqft ~ C(mid_range) * C(buyer_type) + C(sustainability_flag) + '
    'C(emirate) + C(year) + size_sqft + C(property_type) + C(is_offplan)'
)

model3 = smf.ols(formula3, data=df).fit()

print(model3.summary().tables[1])
print(f"\nR-squared: {model3.rsquared:.4f}")

                                                  coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------
Intercept                                       7.0835      0.027    257.964      0.000       7.030       7.137
C(mid_range)[T.1]                              -0.0100      0.023     -0.443      0.658      -0.054       0.034
C(buyer_type)[T.Investor]                       0.2111      0.027      7.712      0.000       0.157       0.265
C(buyer_type)[T.Other]                          0.1424      0.037      3.807      0.000       0.069       0.216
C(sustainability_flag)[T.True]                  0.0873      0.012      7.405      0.000       0.064       0.110
C(emirate)[T.Dubai]                             0.2725      0.012     22.571      0.000       0.249       0.296
C(year)[T.2024]                                 0.1246      0.013      9.488      0.000       0.099     

#### **Model 3 Key Findings & Interpretation**

**Model Summary:**
- Dependent variable: log(price_per_sqft) → coefficients ≈ % change in price per sqft
- R-squared = 0.2049 (improved from ~0.179 in Models 1 & 2) → adding the mid_range × buyer_type interaction explains additional variation
- Observations: 4,000
- Interaction term: mid_range (price AED 0.8–4.5M) × buyer_type

**Statistically Significant Coefficients (p < 0.05):**

- **Mid-Range × Investor Interaction** (C(mid_range)[T.1]:C(buyer_type)[T.Investor]): coef = -0.2234 (p = 0.000)  
  → In the mid-range segment (AED 0.8–4.5M), Investors pay 20% less per sqft compared to what would be expected from the main effects alone (exp(-0.2234) - 1 ≈ -20.0%).  
  → This is a strong negative interaction: mid-range properties attract Investors at lower effective prices per sqft (possibly due to competition or different preferences in this bracket).

- **Mid-Range × Other Interaction** (C(mid_range)[T.1]:C(buyer_type)[T.Other]): coef = -0.1602 (p = 0.000)  
  → Similar pattern: 15% lower price per sqft for Other buyers in mid-range (exp(-0.1602) - 1 ≈ -14.8%).

- **Main Buyer Type Effects** (baseline: First-Time, non-mid-range):  
  - Investor: coef = 0.2111 (p = 0.000) → 23.5% higher price per sqft for Investors outside mid-range  
  - Other: coef = 0.1424 (p = 0.000) → 15.3% higher

- **Sustainability Flag (True)**: coef = 0.0873 (p = 0.000)  
  → Consistent 9.1% premium (exp(0.0873) - 1 ≈ 9.12%), robust across models.

- **Emirate – Dubai**: coef = 0.2725 (p = 0.000)  
  → 31.3% higher price per sqft in Dubai.

- **Year dummies** (vs 2023):  
  - 2024: **0.1246** (p = 0.000) → 13.3% higher  
  - 2025: **0.1835** (p = 0.000) → 20.1% higher

- **Property Type – Townhouse**: coef = 0.0411 (p = 0.039) → 4.2% premium  
- **Size_sqft**: coef = -2.548e-05 (p = 0.012) → very small negative effect (larger properties slightly lower pps, common in per-sqft models)

**Non-significant:**
- Main mid_range effect: coef = -0.0100 (p = 0.658) → no independent effect of being mid-range after interactions
- Off-plan, Villas → no significant impact

**Key Insight from Interaction:**
- The large negative interactions show that Investors and Other buyers pay disproportionately less per sqft in the mid-range segment (where First-Time buyers dominate).  
- This suggests:  
  - Mid-range properties (AED 0.8–4.5M) are a competitive, price-sensitive zone — Investors do not command the same premium here as in luxury segments.  
  - First-Time buyers likely drive volume in this bracket without paying a markup → ideal target for volume-focused, mid-tier developments.

**Business & Portfolio Takeaways:**
- Mid-range segment is First-Time buyer territory — they face less competition from Investors on a per-sqft basis → strong opportunity to target this group with affordable, sustainable apartments/townhouses in Dubai's emerging areas (JVC, Business Bay, etc.).
- Sustainability premium remains stable at 9% across models → continue recommending green/livable features as a reliable value-add.
- Investor behavior shifts in mid-range — they may shift toward off-plan/luxury elsewhere → developers can differentiate mid-range projects by focusing on end-user livability rather than pure investment appeal.
- Improved R² (0.205) shows the interaction captures meaningful buyer-segment dynamics.

This model reinforces the core project recommendation:  
Prioritize mid-income first-time buyers in the AED 0.8–4.5M range with sustainable, community-focused projects in Dubai → leverage consistent 9% sustainability uplift, strong time/emirate trends, and reduced Investor competition in this segment for higher conversion rates and 10–15% potential ROI improvement.